In [1]:
#Importing the basic libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
%matplotlib inline

#Importing libraries necessary to train the model
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.preprocessing import MinMaxScaler #For normalisation of continuous data
from sklearn.preprocessing import OneHotEncoder #For one-hot encoding of host sex, continent and presence/absence of genes
from sklearn.preprocessing import LabelEncoder #For encoding the phenotype
from sklearn.ensemble import RandomForestClassifier 
from sklearn.linear_model import LogisticRegression #Baseline model
from sklearn.metrics import (
    classification_report, roc_curve, auc, precision_recall_curve, 
    average_precision_score, confusion_matrix, accuracy_score,
    f1_score, precision_score, recall_score, brier_score_loss
)
from sklearn.calibration import CalibratedClassifierCV, calibration_curve

from imblearn.pipeline import Pipeline, make_pipeline
from imblearn.over_sampling import SMOTENC

from probatus.feature_elimination import EarlyStoppingShapRFECV

from skopt import BayesSearchCV
from skopt.space import Real, Integer, Categorical

import lightgbm
import xgboost as xgb
from xgboost import XGBClassifier

import seaborn as sns

import shap

import warnings
warnings.filterwarnings("ignore")

/home/micro/miniforge3/envs/shaprfecv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Loading the dataset
data = pd.read_excel("../data/final_dataset.xlsx",
                     sheet_name="Sheet1")
df = data.copy()

#Moving the Label column [Phenotype] to the end of the dataset
temp_cols = df.columns.tolist()
index = df.columns.get_loc("Phenotype")
new_cols = temp_cols[0:index] + temp_cols[index+1:] + temp_cols[index:index+1]
df = df[new_cols]

#Formatting the dataset
df['Continent'] = df['Continent'].replace('America','South America')
df['Phenotype'] = df['Phenotype'].replace({'Gastric cancer ' : 'Gastric cancer', 'Non-gastric cancer ' : 'Non-gastric cancer', 'non-gastric cancer ' : 'Non-gastric cancer' })
df['Sex'] = df['Sex'].str.title()
df.drop(df.index[(df['Sex'] == 'Not Applicable')],axis=0,inplace=True)

In [3]:
X, y = df.iloc[:,1:-1], df.iloc[:,-1] #Splitting the dataframe into Features and Labels

sss = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=26) #Splitting the dataframe into 80% training and 20% test set through StratifiedShuffleSplit
for train_index,test_index in sss.split(X,y):
  X_train_set = X.iloc[train_index]
  y_train_set = y.iloc[train_index]
  X_test_set = X.iloc[test_index]
  y_test_set = y.iloc[test_index]

In [4]:
### Preprocessing the training set

le = LabelEncoder() # Label encoding the Labels in the training set
scaler = MinMaxScaler() # Normalising the continuous data in the training set
encoder = OneHotEncoder(sparse_output=False, handle_unknown="error") # One-hot encoding the categorical data in the training set

y_train_set_encoded = le.fit_transform(y_train_set) # Encoding the labels

numeric_cols = X_train_set.select_dtypes(include=['int64', 'float64']).columns # Selecting the numerical columns
X_train_set[numeric_cols] = scaler.fit_transform(X_train_set[numeric_cols]) # Applying MinMaxScaler only to the numeric columns

X_train_set.loc[:, "homB":"vacAs1m1"] = X_train_set.loc[:, "homB":"vacAs1m1"].astype('object') # Converting these columns to object type
categorical_columns = X_train_set.select_dtypes(include=['object']).columns.tolist() # Selecting the categorical columns

encoder = OneHotEncoder(sparse_output=False, handle_unknown="error") # One-hot encoding the categorical columns
one_hot_encoded = encoder.fit_transform(X_train_set[categorical_columns]) 
one_hot_df = pd.DataFrame(one_hot_encoded, columns=encoder.get_feature_names_out(categorical_columns),
                          index=X_train_set.index) # Creating a DataFrame from the one-hot encoded data

X_train_set_encoded = pd.concat([X_train_set, one_hot_df], axis=1) # Concatenating the one-hot encoded data with the original data

X_train_set_encoded = X_train_set_encoded.drop(categorical_columns, axis=1) # Dropping the original categorical columns
X_train_set_encoded.loc[:, "Sex_Female":"vacAs1m1_1.0"] = X_train_set_encoded.loc[:, "Sex_Female":"vacAs1m1_1.0"].astype('category') # Converting the one-hot encoded columns to category type

In [5]:
### Preprocessing the test set

y_test_set_encoded = le.transform(y_test_set)

numeric_cols_test = X_test_set.select_dtypes(include=['int64', 'float64']).columns # Selecting the numerical columns
X_test_set[numeric_cols_test] = scaler.transform(X_test_set[numeric_cols_test]) # Applying MinMaxScaler only to numeric columns

X_test_set.loc[:, "homB":"vacAs1m1"] = X_test_set.loc[:, "homB":"vacAs1m1"].astype('object') # Converting these columns to object type
categorical_columns_test = X_test_set.select_dtypes(include=['object']).columns.tolist() # Selecting the categorical columns

one_hot_encoded_test = encoder.transform(X_test_set[categorical_columns_test]) # One-hot encoding the categorical columns

one_hot_df_test = pd.DataFrame(one_hot_encoded_test, columns=encoder.get_feature_names_out(categorical_columns_test),
                          index=X_test_set.index) # Creating a DataFrame from the one-hot encoded data

X_test_set_encoded = pd.concat([X_test_set, one_hot_df_test], axis=1) # Concatenating the one-hot encoded data with the original data

X_test_set_encoded = X_test_set_encoded.drop(categorical_columns_test, axis=1) # Dropping the original categorical columns
X_test_set_encoded.loc[:, "Sex_Female":"vacAs1m1_1.0"] = X_test_set_encoded.loc[:, "Sex_Female":"vacAs1m1_1.0"].astype('category') # Converting the one-hot encoded columns to category type

In [6]:
ss = StratifiedShuffleSplit(n_splits=10, random_state=26) # Splitting the dataset into 10 folds for cross-validation

In [7]:
# Logistic Regression

categorical_features = X_train_set_encoded.select_dtypes(include=['category']).columns # Selecting the categorical features
categorical_indices = [X_train_set_encoded.columns.get_loc(col) for col in categorical_features] # Getting the indices of the categorical features for SMOTENC

lr_pipeline = make_pipeline(SMOTENC(categorical_features=categorical_indices,
                                     random_state=26),
                             LogisticRegression(random_state=26)) # Makes a pipeline with SMOTENC and Logistic Regression

lr_pipeline.fit(X_train_set_encoded, y_train_set_encoded) # Fitting the pipeline to the full training set

Pipeline(steps=[('smotenc',
                 SMOTENC(categorical_features=[520, 521, 522, 523, 524, 525,
                                               526, 527, 528, 529, 530, 531,
                                               532, 533, 534, 535, 536, 537,
                                               538, 539, 540, 541, 542, 543,
                                               544, 545, 546, 547],
                         random_state=26)),
                ('logisticregression', LogisticRegression(random_state=26))])

In [ ]:
# Logistic Regression Classifier Evaluation

plt.rcParams.update({           
    'font.size': 8, 'axes.titlesize': 9, 'axes.labelsize': 8,
    'xtick.labelsize': 7, 'ytick.labelsize': 7, 'legend.fontsize': 7,
    'figure.dpi': 1200, 'savefig.dpi': 1200
})

def get_class_info(le, pos_class="Gastric cancer"):
    """Figure out which label is which class"""
    mapping = {name: i for i, name in enumerate(le.classes_)}
    neg_class = [k for k in mapping.keys() if k != pos_class][0]
    
    return {
        'pos_name': pos_class, 'pos_label': mapping[pos_class],
        'neg_name': neg_class, 'neg_label': mapping[neg_class],
        'mapping': mapping
    }

def bootstrap_ci(y_true, y_pred, y_prob, metric_func, n=1000):
    """Get confidence intervals via bootstrap"""
    np.random.seed(26)  # reproducibility
    scores = []
    
    for _ in range(n):
        idx = np.random.choice(len(y_true), len(y_true), replace=True)
        
        # Skip if only one class in bootstrap sample
        if len(np.unique(y_true[idx])) < 2:
            continue
            
        try:
            if y_prob is not None:
                score = metric_func(y_true[idx], y_prob[idx])
            else:
                score = metric_func(y_true[idx], y_pred[idx])
            scores.append(score)
        except:
            continue  # skip problematic samples
    
    if not scores:
        return {'mean': np.nan, 'ci_low': np.nan, 'ci_high': np.nan}
    
    scores = np.array(scores)
    return {
        'mean': np.mean(scores),
        'ci_low': np.percentile(scores, 2.5),
        'ci_high': np.percentile(scores, 97.5)
    }

def plot_calibration(y_test, y_prob_cal, y_prob_raw, class_info, title="Calibration"):
    """Before/after calibration plots"""
    pos_label = class_info['pos_label']
    y_true_binary = (y_test == pos_label).astype(int)
    
    # FIXED: Handle axes creation properly
    if y_prob_raw is not None:
        fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    else:
        fig, ax = plt.subplots(1, 1, figsize=(5, 4))
        axes = [ax]
    
    plot_idx = 0
    
    # Before calibration
    if y_prob_raw is not None:
        frac, pred = calibration_curve(y_true_binary, y_prob_raw[:, pos_label], n_bins=10)
        brier = brier_score_loss(y_true_binary, y_prob_raw[:, pos_label])
        
        axes[plot_idx].plot(pred, frac, 's-', label=class_info['pos_name'])
        axes[plot_idx].plot([0, 1], [0, 1], 'k:', label="Perfect")
        axes[plot_idx].set_title("Before Calibration")
        axes[plot_idx].text(0.02, 0.98, f'Brier Score: {brier:.3f}', 
                           transform=axes[plot_idx].transAxes, va='top',
                           bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
        axes[plot_idx].legend()
        plot_idx += 1
    
    # After calibration
    frac, pred = calibration_curve(y_true_binary, y_prob_cal[:, pos_label], n_bins=10)
    brier = brier_score_loss(y_true_binary, y_prob_cal[:, pos_label])
    
    axes[plot_idx].plot(pred, frac, 's-', label=class_info['pos_name'])
    axes[plot_idx].plot([0, 1], [0, 1], 'k:', label="Perfect")
    axes[plot_idx].set_title("After Calibration" if y_prob_raw is not None else "Calibration")
    axes[plot_idx].text(0.02, 0.98, f'Brier Score: {brier:.3f}', 
                       transform=axes[plot_idx].transAxes, va='top',
                       bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
    axes[plot_idx].legend()
    
    # FIXED: Now axes is always a list, so this works correctly
    for ax in axes:
        ax.set_xlabel("Mean Predicted Probability")
        ax.set_ylabel("Fraction of Positives")
        ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    return fig

def create_metrics_table(y_test, y_pred, y_prob, class_info):
    """Create comprehensive metrics table with confidence intervals"""
    pos_label = class_info['pos_label']
    neg_label = class_info['neg_label']
    pos_name = class_info['pos_name']
    neg_name = class_info['neg_name']
    
    # Calculate metrics with CIs for both classes
    metrics_data = []
    
    # Per-class metrics
    for label, class_name in [(pos_label, pos_name), (neg_label, neg_name)]:
        # Precision
        prec_ci = bootstrap_ci(y_test, y_pred, None, 
                              lambda yt, yp: precision_score(yt, yp, pos_label=label, zero_division=0))
        # Recall (Sensitivity for pos class, Specificity calculation for neg class)
        rec_ci = bootstrap_ci(y_test, y_pred, None, 
                             lambda yt, yp: recall_score(yt, yp, pos_label=label, zero_division=0))
        # F1-Score
        f1_ci = bootstrap_ci(y_test, y_pred, None, 
                            lambda yt, yp: f1_score(yt, yp, pos_label=label, zero_division=0))
        
        # AUPRC for this class
        def ap_func_class(y_true, y_prob):
            return average_precision_score(y_true, y_prob, pos_label=label)
        
        ap_ci = bootstrap_ci(y_test, None, y_prob[:, label], ap_func_class)
        
        metrics_data.extend([
            {
                'Class': class_name,
                'Metric': 'Precision',
                'Value': prec_ci['mean'],
                'CI_Lower': prec_ci['ci_low'],
                'CI_Upper': prec_ci['ci_high'],
                'Final': f"{prec_ci['mean']:.3f} ({prec_ci['ci_low']:.3f}–{prec_ci['ci_high']:.3f})"
            },
            {
                'Class': class_name,
                'Metric': 'Recall',
                'Value': rec_ci['mean'],
                'CI_Lower': rec_ci['ci_low'],
                'CI_Upper': rec_ci['ci_high'],
                'Final': f"{rec_ci['mean']:.3f} ({rec_ci['ci_low']:.3f}–{rec_ci['ci_high']:.3f})"
            },
            {
                'Class': class_name,
                'Metric': 'F1-Score',
                'Value': f1_ci['mean'],
                'CI_Lower': f1_ci['ci_low'],
                'CI_Upper': f1_ci['ci_high'],
                'Final': f"{f1_ci['mean']:.3f} ({f1_ci['ci_low']:.3f}–{f1_ci['ci_high']:.3f})"
            },
            {
                'Class': class_name,
                'Metric': 'AUPRC',
                'Value': ap_ci['mean'],
                'CI_Lower': ap_ci['ci_low'],
                'CI_Upper': ap_ci['ci_high'],
                'Final': f"{ap_ci['mean']:.3f} ({ap_ci['ci_low']:.3f}–{ap_ci['ci_high']:.3f})"
            }
        ])
    
    # Overall metrics
    # Accuracy
    acc_ci = bootstrap_ci(y_test, y_pred, None, lambda yt, yp: accuracy_score(yt, yp))
    
    # AUC-ROC (for positive class)
    def auc_func(y_true, y_prob):
        fpr, tpr, _ = roc_curve(y_true, y_prob, pos_label=pos_label)
        return auc(fpr, tpr)
    auc_ci = bootstrap_ci(y_test, None, y_prob[:, pos_label], auc_func)
    
    # Add overall metrics
    overall_metrics = [
        {
            'Class': 'Overall',
            'Metric': 'Accuracy',
            'Value': acc_ci['mean'],
            'CI_Lower': acc_ci['ci_low'],
            'CI_Upper': acc_ci['ci_high'],
            'Final': f"{acc_ci['mean']:.3f} ({acc_ci['ci_low']:.3f}–{acc_ci['ci_high']:.3f})"
        },
        {
            'Class': 'Overall',
            'Metric': 'AUC-ROC',
            'Value': auc_ci['mean'],
            'CI_Lower': auc_ci['ci_low'],
            'CI_Upper': auc_ci['ci_high'],
            'Final': f"{auc_ci['mean']:.3f} ({auc_ci['ci_low']:.3f}–{auc_ci['ci_high']:.3f})"
        }
    ]
    
    metrics_data.extend(overall_metrics)
    
    # Create DataFrame
    df = pd.DataFrame(metrics_data)
    
    return df

def evaluate_and_plot(model, X_test, y_test, le, save_dir, prefix, 
                      model_name="Model", raw_model=None):
    """
    Full evaluation pipeline:
    - Predictions
    - Bootstrap CIs
    - Metrics table
    - CSV outputs
    - 2x2 grid figure (ROC, PR, metrics, confusion matrix)
    - Calibration plot (separate)
    """
    os.makedirs(f"{save_dir}/figures", exist_ok=True)
    
    class_info = get_class_info(le)
    pos_label = class_info['pos_label']
    neg_label = class_info['neg_label']
    
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)
    y_prob_raw = raw_model.predict_proba(X_test) if raw_model else None
    
    y_test_orig = le.inverse_transform(y_test)
    y_pred_orig = le.inverse_transform(y_pred)
    report = classification_report(y_test_orig, y_pred_orig, output_dict=True)
    pd.DataFrame(report).T.to_csv(f"{save_dir}/{prefix}_report.csv")
    
    def auc_func(y_true, y_prob): return auc(*roc_curve(y_true, y_prob, pos_label=pos_label)[:2])
    def ap_func_pos(y_true, y_prob): return average_precision_score(y_true, y_prob, pos_label=pos_label)
    def ap_func_neg(y_true, y_prob): return average_precision_score(y_true, y_prob, pos_label=neg_label)
    
    auc_ci = bootstrap_ci(y_test, None, y_prob[:, pos_label], auc_func)
    ap_ci_pos = bootstrap_ci(y_test, None, y_prob[:, pos_label], ap_func_pos)
    ap_ci_neg = bootstrap_ci(y_test, None, y_prob[:, neg_label], ap_func_neg)
    
    metrics_table = create_metrics_table(y_test, y_pred, y_prob, class_info)
    metrics_table.to_csv(f"{save_dir}/{prefix}_detailed_metrics.csv", index=False)
    
    fig, axes = plt.subplots(2, 2, figsize=(18, 14))
    plt.subplots_adjust(hspace=0.7, wspace=0.7)

    fontsize_labels = 16   
    fontsize_ticks = 14    
    fontsize_legend = 14   
    fontsize_title = 18    
    
    fpr, tpr, _ = roc_curve(y_test, y_prob[:, pos_label], pos_label=pos_label)
    axes[0,0].plot(fpr, tpr, 'r-', linewidth=2,
                   label=f"{class_info['pos_name']} - AUC = {auc_ci['mean']:.3f} "
                         f"(95% CI: {auc_ci['ci_low']:.3f}–{auc_ci['ci_high']:.3f})")
    axes[0,0].plot([0,1],[0,1],'k--', alpha=0.5)
    axes[0,0].set_xlabel("False Positive Rate", fontsize=fontsize_labels)
    axes[0,0].set_ylabel("True Positive Rate", fontsize=fontsize_labels)
    axes[0,0].tick_params(axis='both', which='major', labelsize=fontsize_ticks)
    axes[0,0].set_title("A. ROC Curve", fontsize=fontsize_title, fontweight='bold')
    axes[0,0].legend(fontsize=fontsize_legend)
    axes[0,0].grid(True, alpha=0.3)
    

    prec, rec, _ = precision_recall_curve(y_test, y_prob[:, pos_label], pos_label=pos_label)
    prec_neg, rec_neg, _ = precision_recall_curve(y_test, y_prob[:, neg_label], pos_label=neg_label)
    axes[0,1].plot(rec, prec, 'b-', linewidth=2,
                   label=f"{class_info['pos_name']} - AUPRC = {ap_ci_pos['mean']:.3f} "
                         f"(95% CI: {ap_ci_pos['ci_low']:.3f}–{ap_ci_pos['ci_high']:.3f})")
    axes[0,1].plot(rec_neg, prec_neg, 'g--', linewidth=2,
                   label=f"{class_info['neg_name']} - AUPRC = {ap_ci_neg['mean']:.3f} "
                         f"(95% CI: {ap_ci_neg['ci_low']:.3f}–{ap_ci_neg['ci_high']:.3f})")
    axes[0,1].set_xlabel("Recall", fontsize=fontsize_labels)
    axes[0,1].set_ylabel("Precision", fontsize=fontsize_labels)
    axes[0,1].set_title("B. Precision-Recall Curve", fontsize=fontsize_title, fontweight='bold')
    axes[0,1].tick_params(axis='both', which='major', labelsize=fontsize_ticks)
    axes[0,1].legend(fontsize=fontsize_legend)
    axes[0,1].grid(True, alpha=0.3)
    

    metrics = {}
    for label, name in [(pos_label, class_info['pos_name']), (neg_label, class_info['neg_name'])]:
        metrics[name] = {
            'Precision': bootstrap_ci(y_test, y_pred, None, lambda yt, yp: precision_score(yt, yp, pos_label=label, zero_division=0)),
            'Recall': bootstrap_ci(y_test, y_pred, None, lambda yt, yp: recall_score(yt, yp, pos_label=label, zero_division=0)),
            'F1-Score': bootstrap_ci(y_test, y_pred, None, lambda yt, yp: f1_score(yt, yp, pos_label=label, zero_division=0))
        }
    metric_names = ['Precision', 'Recall', 'F1-Score']
    class_names = list(metrics.keys())
    pos_means = [metrics[class_names[0]][m]['mean'] for m in metric_names]
    neg_means = [metrics[class_names[1]][m]['mean'] for m in metric_names]
    pos_errs = [(metrics[class_names[0]][m]['ci_high'] - metrics[class_names[0]][m]['ci_low'])/2 for m in metric_names]
    neg_errs = [(metrics[class_names[1]][m]['ci_high'] - metrics[class_names[1]][m]['ci_low'])/2 for m in metric_names]
    
    y_pos = np.arange(len(metric_names))
    width = 0.35
    axes[1,0].barh(y_pos - width/2, pos_means, width, xerr=pos_errs, label=class_names[0], color='#E31A1C', alpha=0.8, capsize=3, hatch='///')
    axes[1,0].barh(y_pos + width/2, neg_means, width, xerr=neg_errs, label=class_names[1], color='#1F78B4', alpha=0.8, capsize=3, hatch='...')
    
    for i in range(len(metric_names)):
        axes[1,0].text(pos_means[i]+pos_errs[i]+0.02, y_pos[i]-width/2, f'{pos_means[i]:.3f}', va='center', fontsize=10)
        axes[1,0].text(neg_means[i]+neg_errs[i]+0.02, y_pos[i]+width/2, f'{neg_means[i]:.3f}', va='center', fontsize=10)
    
    axes[1,0].set_yticks(y_pos)
    axes[1,0].set_yticklabels(metric_names)
    axes[1,0].set_xlabel("Score", fontsize=fontsize_labels)
    axes[1,0].set_title("C. Classification Metrics", fontsize=fontsize_title, fontweight='bold')
    axes[1,0].tick_params(axis='both', which='major', labelsize=fontsize_ticks)
    axes[1,0].legend(fontsize=fontsize_legend)
    axes[1,0].grid(True, alpha=0.3, axis='x')
    axes[1,0].set_xlim(0, 1.15)
    

    labels = [neg_label, pos_label]
    names = [class_info['neg_name'], class_info['pos_name']]
    cm = confusion_matrix(y_test, y_pred, labels=labels)
    cm_pct = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis] * 100
    sns.heatmap(cm, annot=False, cmap='Blues', xticklabels=names, yticklabels=names, ax=axes[1,1])
    
    for i in range(2):
        for j in range(2):
            color = 'white' if cm[i,j] > cm.max()/2 else 'black'
            axes[1,1].text(j+0.5, i+0.5, f'{cm[i,j]:,}\n({cm_pct[i,j]:.1f}%)', ha='center', va='center', color=color, fontweight='bold')
    
    axes[1,1].set_title("D. Confusion Matrix", fontsize=fontsize_title, fontweight='bold')
    axes[1,1].set_xlabel("Predicted Labels", fontsize=fontsize_labels)
    axes[1,1].set_ylabel("True Labels", fontsize=fontsize_labels)
    axes[1,1].tick_params(axis='both', which='major', labelsize=fontsize_ticks)
    
    tn, fp, fn, tp = cm.ravel()
    acc = (tp + tn)/cm.sum()
    sens = tp/(tp+fn) if (tp+fn)>0 else 0
    spec = tn/(tn+fp) if (tn+fp)>0 else 0
    axes[1,1].text(0.5, -0.18, f'Accuracy: {acc:.3f} | Sensitivity: {sens:.3f} | Specificity: {spec:.3f}', 
                    transform=axes[1,1].transAxes, ha='center', style='italic', fontsize=12)
    
    fig.suptitle(f"{model_name} Evaluation", fontsize=20, fontweight='bold')
    plt.tight_layout(rect=[0,0,1,0.96])
    

    fig.savefig(f"{save_dir}/figures/{prefix}_2x2_grid.png", dpi=1200, bbox_inches='tight')
    fig.savefig(f"{save_dir}/figures/{prefix}_2x2_grid.pdf", bbox_inches='tight')
    plt.close(fig)
    

    fig_cal = plot_calibration(y_test, y_prob, y_prob_raw, class_info)
    fig_cal.savefig(f"{save_dir}/figures/{prefix}_calibration.png", dpi=1200, bbox_inches='tight')
    fig_cal.savefig(f"{save_dir}/figures/{prefix}_calibration.pdf", bbox_inches='tight')
    plt.close(fig_cal)
    

    return {
        'y_pred': y_pred,
        'y_prob': y_prob,
        'auc': auc_ci,
        'auprc_pos': ap_ci_pos,
        'auprc_neg': ap_ci_neg,
        'metrics_table': metrics_table,
        'fig_grid': fig,
        'fig_calibration': fig_cal
    }

evaluate_and_plot(
    model=lr_pipeline,         
    X_test=X_test_set_encoded,             
    y_test=y_test_set_encoded,         
    le=le,                             
    save_dir="../results/LR",  
    prefix="LR",                        
    model_name="Logistic Regression",        
    raw_model=None          
)

Using classifier: logisticregression (LogisticRegression)
Computing SHAP values...
SHAP summary plot saved to: ../results/2025-09-09_v1_LR/figures/shap_summary_plot.png
Sample 2 predicted as: Gastric cancer
SHAP waterfall plot saved to: ../results/2025-09-09_v1_LR/figures/shap_waterfall_sample_2.png


In [ ]:
# Logistic Regression Explanation with SHAP

def explain_model_with_shap_plots(
    pipeline, X_train, X_test, save_dir, classifier_step_name="classifier", 
    sample_idx=0, only_explain_gc=True
):
    """
    Generates SHAP global summary plot and SHAP waterfall plot for a specific sample.
    """
    # Set larger font sizes for all plots
    plt.rcParams.update({
        'font.size': 14,
        'axes.titlesize': 16,
        'axes.labelsize': 14,
        'xtick.labelsize': 12,
        'ytick.labelsize': 12,
        'legend.fontsize': 12
    })

    os.makedirs(os.path.join(save_dir, "figures"), exist_ok=True)

    # Convert categorical variables to numeric
    X_train_num = X_train.copy()
    X_test_num = X_test.copy()
    
    for col in X_train_num.select_dtypes(include="category").columns:
        X_train_num[col] = X_train_num[col].cat.codes
        X_test_num[col] = X_test_num[col].cat.codes

    X_train_num = X_train_num.astype(float)
    X_test_num = X_test_num.astype(float)
    
    # Handle problematic values
    X_train_num = X_train_num.replace([np.inf, -np.inf], np.nan)
    X_test_num = X_test_num.replace([np.inf, -np.inf], np.nan)
    X_train_num = X_train_num.fillna(X_train_num.median())
    X_test_num = X_test_num.fillna(X_train_num.median())

    # Auto-detect classifier step
    if hasattr(pipeline, "named_steps"):  
        model_step_names = list(pipeline.named_steps.keys())
        classifier_step_name = model_step_names[-1]
        model = pipeline.named_steps[classifier_step_name]
    elif hasattr(pipeline, "estimator") and hasattr(pipeline.estimator, "named_steps"):  
        model_step_names = list(pipeline.estimator.named_steps.keys())
        classifier_step_name = model_step_names[-1]
        model = pipeline.estimator.named_steps[classifier_step_name]
    else:
        raise ValueError("Could not find classifier step in pipeline.")

    print(f"Using classifier: {classifier_step_name} ({type(model).__name__})")

    # Select appropriate SHAP explainer
    if isinstance(model, (RandomForestClassifier, xgb.XGBClassifier)):
        explainer = shap.TreeExplainer(model)
    elif isinstance(model, LogisticRegression):
        background_sample = shap.sample(X_train_num, 100)
        explainer = shap.LinearExplainer(model, background_sample)
    else:
        background_sample = shap.sample(X_train_num, 50)
        explainer = shap.PermutationExplainer(model.predict, background_sample)

    # Calculate SHAP values
    print("Computing SHAP values...")
    shap_values = explainer(X_test_num)

    if sample_idx >= len(X_test_num):
        sample_idx = 0
    
    sample_pred = pipeline.predict(X_test_num.iloc[[sample_idx]])[0]
    sample_probs = pipeline.predict_proba(X_test_num.iloc[[sample_idx]])[0]

    predicted_class = "Gastric cancer" if sample_pred == 0 else "Non-gastric cancer"
    print(f"Sample {sample_idx} predicted as: {predicted_class}")

    if only_explain_gc and sample_pred != 0:
        print(f"Skipping explanation: Sample not classified as gastric cancer.")
        return

    # Handle SHAP value formats and prepare data
    if hasattr(shap_values, 'values'):
        if shap_values.values.ndim == 3:
            values_for_bar = shap_values.values[:, :, 0]
            waterfall_values = shap_values.values[sample_idx, :, 0]
            expected_value = shap_values.base_values[sample_idx, 0]
        else:
            values_for_bar = shap_values.values
            waterfall_values = shap_values.values[sample_idx]
            expected_value = shap_values.base_values[sample_idx]
        feature_values = shap_values.data
    else:
        values_for_bar = shap_values
        waterfall_values = shap_values[sample_idx]
        expected_value = explainer.expected_value
        feature_values = X_test_num.values

    # Create SHAP explanation object for bar plot
    shap_explanation_for_bar = shap.Explanation(
        values=values_for_bar,
        base_values=expected_value,
        data=feature_values,
        feature_names=X_test_num.columns.tolist()
    )

    # Create explanation object for waterfall plot
    explanation_obj = shap.Explanation(
        values=waterfall_values,
        base_values=expected_value,
        data=X_test_num.iloc[sample_idx].values,
        feature_names=X_test_num.columns.tolist()
    )

    # Create combined plot with subplots
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(20, 40), gridspec_kw={'hspace': 0.8})
    
    # Global SHAP bar plot
    ax1.text(-0.02, 1.25, 'A', transform=ax1.transAxes, fontsize=20, fontweight='bold', va='top')
    shap.plots.bar(shap_explanation_for_bar, ax=ax1, show=False, max_display=10)
    ax1.set_title('Global Feature Importance', fontsize=16, fontweight='bold', pad=20)
    
    # Individual waterfall plot 
    ax2.text(-0.02, 1.4, 'B', transform=ax2.transAxes, fontsize=20, fontweight='bold', va='top')
    plt.sca(ax2)
    shap.waterfall_plot(explanation_obj, show=False)
    ax2.set_title(f'Individual Prediction Explanation - Sample {sample_idx}', fontsize=16, fontweight='bold', pad=20)
    
    plt.tight_layout(pad=2.0)
    combined_path = os.path.join(save_dir, "figures", f"shap_combined_plots_sample_{sample_idx}.png")
    plt.savefig(combined_path, dpi=1200, bbox_inches="tight", facecolor="white")
    plt.close()
    print(f"Combined SHAP plots saved to: {combined_path}")



explain_model_with_shap_plots(
    lr_pipeline, 
    X_train_set_encoded, 
    X_test_set_encoded, 
    "../results/LR", 
    sample_idx=2,
    only_explain_gc=True
)

Using classifier: logisticregression (LogisticRegression)
Computing SHAP values...
Sample 2 predicted as: Gastric cancer
Combined SHAP plots saved to: ../results/LR/figures/shap_combined_plots_sample_2.png


In [ ]:
# Bayesian Optimization and Feature Elimination

model_bayes = lightgbm.LGBMClassifier(max_depth=5, class_weight="balanced") # Model for Bayesian optimization

param_grid_bayes = {                 # Hyperparameter grid for Bayesian optimization
    "n_estimators": [5, 7, 10],
    "num_leaves": [3, 5, 7, 10],
}

search_bayes = BayesSearchCV(       # Bayesian optimization using scikit-optimize
    estimator=model_bayes,
    search_spaces=param_grid_bayes,
    verbose=0,
    random_state=26
)

# Create the EarlyStoppingShapRFECV object with early stopping parameters
shap_elimination_bayes = EarlyStoppingShapRFECV(
    model=search_bayes,           # hyperparameter tuning wrapper
    step=0.2,                     # proportion or number of features to remove per iteration
    cv=ss,                        # cross-validation splitter
    scoring='recall',             # evaluation metric used for CV
    eval_metric='recall',         # early stopping evaluation metric
    early_stopping_rounds=30,     # stop if no improvement is seen for 30 rounds
    n_jobs=-1,
    verbose=-1,
    random_state=26
)

# Run feature selection
report_bayes = shap_elimination_bayes.fit_compute(X_train_set_encoded, y_train_set_encoded)

best_features_bayes = shap_elimination_bayes.get_reduced_features_set(num_features="best") # Get the best features

final_features_set_bayes = shap_elimination_bayes.get_reduced_features_set(num_features=len(best_features_bayes)) # Get the final features set

[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Number of positive: 605, number of negative: 266
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.005498 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 94118
[LightGBM] [Info] Number of data points in the train set: 871, number of used features: 498
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=-0.000000
[LightGBM] [Info] Start training from score -0.000000
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Number of positive: 605, number of negative: 266
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.008500 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 94087
[LightGBM] [Info] Number of data points in the train set: 871, number of used features

Exception ignored in: <function ResourceTracker.__del__ at 0x7b3a83a4c540>
Traceback (most recent call last):
  File "/home/micro/miniforge3/envs/shaprfecv/lib/python3.12/multiprocessing/resource_tracker.py", line 77, in __del__
  File "/home/micro/miniforge3/envs/shaprfecv/lib/python3.12/multiprocessing/resource_tracker.py", line 86, in _stop
  File "/home/micro/miniforge3/envs/shaprfecv/lib/python3.12/multiprocessing/resource_tracker.py", line 111, in _stop_locked
ChildProcessError: [Errno 10] No child processes
Exception ignored in: <function ResourceTracker.__del__ at 0x74925344c540>
Traceback (most recent call last):
  File "/home/micro/miniforge3/envs/shaprfecv/lib/python3.12/multiprocessing/resource_tracker.py", line 77, in __del__
  File "/home/micro/miniforge3/envs/shaprfecv/lib/python3.12/multiprocessing/resource_tracker.py", line 86, in _stop
  File "/home/micro/miniforge3/envs/shaprfecv/lib/python3.12/multiprocessing/resource_tracker.py", line 111, in _stop_locked
ChildProc

In [19]:
final_features_set_bayes

['Age',
 'conservative_inframe_insertion',
 'disruptive_inframe_deletion',
 'disruptive_inframe_deletion&synonymous_variant',
 'frameshift_variant',
 'frameshift_variant&start_lost',
 'frameshift_variant&stop_gained',
 'frameshift_variant&stop_lost',
 'frameshift_variant&stop_lost&missense_variant&splice_region_variant',
 'frameshift_variant&stop_lost&splice_region_variant',
 'frameshift_variant&stop_lost&splice_region_variant&stop_retained_variant',
 'frameshift_variant&stop_lost&stop_retained_variant&splice_region_variant',
 'frameshift_variant&synonymous_variant',
 'intergenic_region',
 'intragenic_variant',
 'missense_variant',
 'missense_variant&conservative_inframe_insertion',
 'missense_variant&disruptive_inframe_deletion',
 'splice_region_variant&stop_retained_variant',
 'start_lost',
 'start_lost&missense_variant',
 'stop_gained',
 'stop_gained&conservative_inframe_insertion',
 'stop_gained&missense_variant&conservative_inframe_insertion',
 'stop_lost',
 'stop_retained_variant

In [20]:
X_train_reduced = X_train_set_encoded[final_features_set_bayes] # Subset the training set to the final features set
X_test_reduced = X_test_set_encoded[final_features_set_bayes] # Subset the test set to the final features set

categorical_features_subset = X_train_reduced.select_dtypes(include=['category']).columns # Selecting the categorical features from the reduced training set
categorical_indices_subset = [X_train_reduced.columns.get_loc(col) for col in categorical_features_subset] # Getting the indices of the categorical features for SMOTENC

In [21]:
# XGBoost Classifier

xgb_pipeline = Pipeline([
    ('smote', SMOTENC(categorical_features=categorical_indices_subset, random_state=26)),
    ('classifier', XGBClassifier(
        objective='binary:logistic',
        tree_method='hist',
        enable_categorical=True,
        eval_metric='logloss',
        random_state=26
    ))
])


xgb_search_space = {                      
    'classifier__n_estimators': Integer(50, 1000),
    'classifier__max_depth': Integer(3, 7),
    'classifier__learning_rate': Real(0.01, 0.1),
    'classifier__subsample': Real(0.5, 1),
    'classifier__colsample_bytree': Real(0.5, 1),
    'classifier__min_child_weight': Integer(1, 10),
    'classifier__gamma': Real(0, 5),
    'classifier__reg_alpha': Real(1e-9, 10.0, prior='log-uniform'),
    'classifier__reg_lambda': Real(1e-9, 10.0, prior='log-uniform'),
}


xgb_bayes_search = BayesSearchCV(      # Bayesian optimization using scikit-optimize
    estimator=xgb_pipeline,
    search_spaces=xgb_search_space,
    scoring='recall',  # Using F1 score as the evaluation metric
    cv=ss,
    n_iter=60,
    n_points=2,
    n_jobs=2,
    verbose=2,
    random_state=26
)


xgb_bayes_search.fit(X_train_reduced, y_train_set_encoded) # Fit the XGBClassifier model on the reduced training set

xgb_calibrated_model = CalibratedClassifierCV(xgb_bayes_search.best_estimator_, method='sigmoid', cv=ss) # Calibrating the model using the best estimator from Bayesian optimization
xgb_calibrated_model.fit(X_train_reduced, y_train_set_encoded)

Fitting 10 folds for each of 2 candidates, totalling 20 fits
[CV] END classifier__colsample_bytree=0.8710253864103497, classifier__gamma=4.937741895069575, classifier__learning_rate=0.03328017103854301, classifier__max_depth=4, classifier__min_child_weight=6, classifier__n_estimators=970, classifier__reg_alpha=0.00018075619344602056, classifier__reg_lambda=0.00016473374731881883, classifier__subsample=0.7273233990541943; total time=   1.3s
[CV] END classifier__colsample_bytree=0.8710253864103497, classifier__gamma=4.937741895069575, classifier__learning_rate=0.03328017103854301, classifier__max_depth=4, classifier__min_child_weight=6, classifier__n_estimators=970, classifier__reg_alpha=0.00018075619344602056, classifier__reg_lambda=0.00016473374731881883, classifier__subsample=0.7273233990541943; total time=   1.3s
[CV] END classifier__colsample_bytree=0.8710253864103497, classifier__gamma=4.937741895069575, classifier__learning_rate=0.03328017103854301, classifier__max_depth=4, classi

CalibratedClassifierCV(cv=StratifiedShuffleSplit(n_splits=10, random_state=26, test_size=None,
            train_size=None),
                       estimator=Pipeline(steps=[('smote',
                                                  SMOTENC(categorical_features=[92],
                                                          random_state=26)),
                                                 ('classifier',
                                                  XGBClassifier(base_score=None,
                                                                booster=None,
                                                                callbacks=None,
                                                                colsample_bylevel=None,
                                                                colsample_bynode=None,
                                                                colsample_bytree=1.0,
                                                                device=None,
                                                                earl...
                                                                gamma=0.0,
                                                                grow_policy=None,
                                                                importance_type=None,
                                                                interaction_constraints=None,
                                                                learning_rate=0.027929298857299313,
                                                                max_bin=None,
                                                                max_cat_threshold=None,
                                                                max_cat_to_onehot=None,
                                                                max_delta_step=None,
                                                                max_depth=7,
                                                                max_leaves=None,
                                                                min_child_weight=1,
                                                                missing=nan,
                                                                monotone_constraints=None,
                                                                multi_strategy=None,
                                                                n_estimators=1000,
                                                                n_jobs=None,
                                                                num_parallel_tree=None, ...))]))

In [ ]:
# XGBoost Classifier Evaluation

plt.rcParams.update({           
    'font.size': 8, 'axes.titlesize': 9, 'axes.labelsize': 8,
    'xtick.labelsize': 7, 'ytick.labelsize': 7, 'legend.fontsize': 7,
    'figure.dpi': 1200, 'savefig.dpi': 1200
})

def get_class_info(le, pos_class="Gastric cancer"):
    """Figure out which label is which class"""
    mapping = {name: i for i, name in enumerate(le.classes_)}
    neg_class = [k for k in mapping.keys() if k != pos_class][0]
    
    return {
        'pos_name': pos_class, 'pos_label': mapping[pos_class],
        'neg_name': neg_class, 'neg_label': mapping[neg_class],
        'mapping': mapping
    }

def bootstrap_ci(y_true, y_pred, y_prob, metric_func, n=1000):
    """Get confidence intervals via bootstrap"""
    np.random.seed(26)  # reproducibility
    scores = []
    
    for _ in range(n):
        idx = np.random.choice(len(y_true), len(y_true), replace=True)
        
        # Skip if only one class in bootstrap sample
        if len(np.unique(y_true[idx])) < 2:
            continue
            
        try:
            if y_prob is not None:
                score = metric_func(y_true[idx], y_prob[idx])
            else:
                score = metric_func(y_true[idx], y_pred[idx])
            scores.append(score)
        except:
            continue  # skip problematic samples
    
    if not scores:
        return {'mean': np.nan, 'ci_low': np.nan, 'ci_high': np.nan}
    
    scores = np.array(scores)
    return {
        'mean': np.mean(scores),
        'ci_low': np.percentile(scores, 2.5),
        'ci_high': np.percentile(scores, 97.5)
    }

def plot_calibration(y_test, y_prob_cal, y_prob_raw, class_info, title="Calibration"):
    """Before/after calibration plots"""
    pos_label = class_info['pos_label']
    y_true_binary = (y_test == pos_label).astype(int)
    
    # FIXED: Handle axes creation properly
    if y_prob_raw is not None:
        fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    else:
        fig, ax = plt.subplots(1, 1, figsize=(5, 4))
        axes = [ax]
    
    plot_idx = 0
    
    # Before calibration
    if y_prob_raw is not None:
        frac, pred = calibration_curve(y_true_binary, y_prob_raw[:, pos_label], n_bins=10)
        brier = brier_score_loss(y_true_binary, y_prob_raw[:, pos_label])
        
        axes[plot_idx].plot(pred, frac, 's-', label=class_info['pos_name'])
        axes[plot_idx].plot([0, 1], [0, 1], 'k:', label="Perfect")
        axes[plot_idx].set_title("Before Calibration")
        axes[plot_idx].text(0.02, 0.98, f'Brier Score: {brier:.3f}', 
                           transform=axes[plot_idx].transAxes, va='top',
                           bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
        axes[plot_idx].legend()
        plot_idx += 1
    
    # After calibration
    frac, pred = calibration_curve(y_true_binary, y_prob_cal[:, pos_label], n_bins=10)
    brier = brier_score_loss(y_true_binary, y_prob_cal[:, pos_label])
    
    axes[plot_idx].plot(pred, frac, 's-', label=class_info['pos_name'])
    axes[plot_idx].plot([0, 1], [0, 1], 'k:', label="Perfect")
    axes[plot_idx].set_title("After Calibration" if y_prob_raw is not None else "Calibration")
    axes[plot_idx].text(0.02, 0.98, f'Brier Score: {brier:.3f}', 
                       transform=axes[plot_idx].transAxes, va='top',
                       bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
    axes[plot_idx].legend()
    
    # FIXED: Now axes is always a list, so this works correctly
    for ax in axes:
        ax.set_xlabel("Mean Predicted Probability")
        ax.set_ylabel("Fraction of Positives")
        ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    return fig

def create_metrics_table(y_test, y_pred, y_prob, class_info):
    """Create comprehensive metrics table with confidence intervals"""
    pos_label = class_info['pos_label']
    neg_label = class_info['neg_label']
    pos_name = class_info['pos_name']
    neg_name = class_info['neg_name']
    
    # Calculate metrics with CIs for both classes
    metrics_data = []
    
    # Per-class metrics
    for label, class_name in [(pos_label, pos_name), (neg_label, neg_name)]:
        # Precision
        prec_ci = bootstrap_ci(y_test, y_pred, None, 
                              lambda yt, yp: precision_score(yt, yp, pos_label=label, zero_division=0))
        # Recall (Sensitivity for pos class, Specificity calculation for neg class)
        rec_ci = bootstrap_ci(y_test, y_pred, None, 
                             lambda yt, yp: recall_score(yt, yp, pos_label=label, zero_division=0))
        # F1-Score
        f1_ci = bootstrap_ci(y_test, y_pred, None, 
                            lambda yt, yp: f1_score(yt, yp, pos_label=label, zero_division=0))
        
        # AUPRC for this class
        def ap_func_class(y_true, y_prob):
            return average_precision_score(y_true, y_prob, pos_label=label)
        
        ap_ci = bootstrap_ci(y_test, None, y_prob[:, label], ap_func_class)
        
        metrics_data.extend([
            {
                'Class': class_name,
                'Metric': 'Precision',
                'Value': prec_ci['mean'],
                'CI_Lower': prec_ci['ci_low'],
                'CI_Upper': prec_ci['ci_high'],
                'Final': f"{prec_ci['mean']:.3f} ({prec_ci['ci_low']:.3f}–{prec_ci['ci_high']:.3f})"
            },
            {
                'Class': class_name,
                'Metric': 'Recall',
                'Value': rec_ci['mean'],
                'CI_Lower': rec_ci['ci_low'],
                'CI_Upper': rec_ci['ci_high'],
                'Final': f"{rec_ci['mean']:.3f} ({rec_ci['ci_low']:.3f}–{rec_ci['ci_high']:.3f})"
            },
            {
                'Class': class_name,
                'Metric': 'F1-Score',
                'Value': f1_ci['mean'],
                'CI_Lower': f1_ci['ci_low'],
                'CI_Upper': f1_ci['ci_high'],
                'Final': f"{f1_ci['mean']:.3f} ({f1_ci['ci_low']:.3f}–{f1_ci['ci_high']:.3f})"
            },
            {
                'Class': class_name,
                'Metric': 'AUPRC',
                'Value': ap_ci['mean'],
                'CI_Lower': ap_ci['ci_low'],
                'CI_Upper': ap_ci['ci_high'],
                'Final': f"{ap_ci['mean']:.3f} ({ap_ci['ci_low']:.3f}–{ap_ci['ci_high']:.3f})"
            }
        ])
    
    # Overall metrics
    # Accuracy
    acc_ci = bootstrap_ci(y_test, y_pred, None, lambda yt, yp: accuracy_score(yt, yp))
    
    # AUC-ROC (for positive class)
    def auc_func(y_true, y_prob):
        fpr, tpr, _ = roc_curve(y_true, y_prob, pos_label=pos_label)
        return auc(fpr, tpr)
    auc_ci = bootstrap_ci(y_test, None, y_prob[:, pos_label], auc_func)
    
    # Add overall metrics
    overall_metrics = [
        {
            'Class': 'Overall',
            'Metric': 'Accuracy',
            'Value': acc_ci['mean'],
            'CI_Lower': acc_ci['ci_low'],
            'CI_Upper': acc_ci['ci_high'],
            'Final': f"{acc_ci['mean']:.3f} ({acc_ci['ci_low']:.3f}–{acc_ci['ci_high']:.3f})"
        },
        {
            'Class': 'Overall',
            'Metric': 'AUC-ROC',
            'Value': auc_ci['mean'],
            'CI_Lower': auc_ci['ci_low'],
            'CI_Upper': auc_ci['ci_high'],
            'Final': f"{auc_ci['mean']:.3f} ({auc_ci['ci_low']:.3f}–{auc_ci['ci_high']:.3f})"
        }
    ]
    
    metrics_data.extend(overall_metrics)
    
    # Create DataFrame
    df = pd.DataFrame(metrics_data)
    
    return df

def evaluate_and_plot(model, X_test, y_test, le, save_dir, prefix, 
                      model_name="Model", raw_model=None):
    """
    Full evaluation pipeline:
    - Predictions
    - Bootstrap CIs
    - Metrics table
    - CSV outputs
    - 2x2 grid figure (ROC, PR, metrics, confusion matrix)
    - Calibration plot (separate)
    """
    os.makedirs(f"{save_dir}/figures", exist_ok=True)
    
    class_info = get_class_info(le)
    pos_label = class_info['pos_label']
    neg_label = class_info['neg_label']
    
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)
    y_prob_raw = raw_model.predict_proba(X_test) if raw_model else None
    
    y_test_orig = le.inverse_transform(y_test)
    y_pred_orig = le.inverse_transform(y_pred)
    report = classification_report(y_test_orig, y_pred_orig, output_dict=True)
    pd.DataFrame(report).T.to_csv(f"{save_dir}/{prefix}_report.csv")
    
    def auc_func(y_true, y_prob): return auc(*roc_curve(y_true, y_prob, pos_label=pos_label)[:2])
    def ap_func_pos(y_true, y_prob): return average_precision_score(y_true, y_prob, pos_label=pos_label)
    def ap_func_neg(y_true, y_prob): return average_precision_score(y_true, y_prob, pos_label=neg_label)
    
    auc_ci = bootstrap_ci(y_test, None, y_prob[:, pos_label], auc_func)
    ap_ci_pos = bootstrap_ci(y_test, None, y_prob[:, pos_label], ap_func_pos)
    ap_ci_neg = bootstrap_ci(y_test, None, y_prob[:, neg_label], ap_func_neg)
    
    metrics_table = create_metrics_table(y_test, y_pred, y_prob, class_info)
    metrics_table.to_csv(f"{save_dir}/{prefix}_detailed_metrics.csv", index=False)
    
    fig, axes = plt.subplots(2, 2, figsize=(18, 14))
    plt.subplots_adjust(hspace=0.7, wspace=0.7)

    fontsize_labels = 16   
    fontsize_ticks = 14    
    fontsize_legend = 14   
    fontsize_title = 18    
    
    fpr, tpr, _ = roc_curve(y_test, y_prob[:, pos_label], pos_label=pos_label)
    axes[0,0].plot(fpr, tpr, 'r-', linewidth=2,
                   label=f"{class_info['pos_name']} - AUC = {auc_ci['mean']:.3f} "
                         f"(95% CI: {auc_ci['ci_low']:.3f}–{auc_ci['ci_high']:.3f})")
    axes[0,0].plot([0,1],[0,1],'k--', alpha=0.5)
    axes[0,0].set_xlabel("False Positive Rate", fontsize=fontsize_labels)
    axes[0,0].set_ylabel("True Positive Rate", fontsize=fontsize_labels)
    axes[0,0].tick_params(axis='both', which='major', labelsize=fontsize_ticks)
    axes[0,0].set_title("A. ROC Curve", fontsize=fontsize_title, fontweight='bold')
    axes[0,0].legend(fontsize=fontsize_legend)
    axes[0,0].grid(True, alpha=0.3)
    

    prec, rec, _ = precision_recall_curve(y_test, y_prob[:, pos_label], pos_label=pos_label)
    prec_neg, rec_neg, _ = precision_recall_curve(y_test, y_prob[:, neg_label], pos_label=neg_label)
    axes[0,1].plot(rec, prec, 'b-', linewidth=2,
                   label=f"{class_info['pos_name']} - AUPRC = {ap_ci_pos['mean']:.3f} "
                         f"(95% CI: {ap_ci_pos['ci_low']:.3f}–{ap_ci_pos['ci_high']:.3f})")
    axes[0,1].plot(rec_neg, prec_neg, 'g--', linewidth=2,
                   label=f"{class_info['neg_name']} - AUPRC = {ap_ci_neg['mean']:.3f} "
                         f"(95% CI: {ap_ci_neg['ci_low']:.3f}–{ap_ci_neg['ci_high']:.3f})")
    axes[0,1].set_xlabel("Recall", fontsize=fontsize_labels)
    axes[0,1].set_ylabel("Precision", fontsize=fontsize_labels)
    axes[0,1].set_title("B. Precision-Recall Curve", fontsize=fontsize_title, fontweight='bold')
    axes[0,1].tick_params(axis='both', which='major', labelsize=fontsize_ticks)
    axes[0,1].legend(fontsize=fontsize_legend)
    axes[0,1].grid(True, alpha=0.3)
    

    metrics = {}
    for label, name in [(pos_label, class_info['pos_name']), (neg_label, class_info['neg_name'])]:
        metrics[name] = {
            'Precision': bootstrap_ci(y_test, y_pred, None, lambda yt, yp: precision_score(yt, yp, pos_label=label, zero_division=0)),
            'Recall': bootstrap_ci(y_test, y_pred, None, lambda yt, yp: recall_score(yt, yp, pos_label=label, zero_division=0)),
            'F1-Score': bootstrap_ci(y_test, y_pred, None, lambda yt, yp: f1_score(yt, yp, pos_label=label, zero_division=0))
        }
    metric_names = ['Precision', 'Recall', 'F1-Score']
    class_names = list(metrics.keys())
    pos_means = [metrics[class_names[0]][m]['mean'] for m in metric_names]
    neg_means = [metrics[class_names[1]][m]['mean'] for m in metric_names]
    pos_errs = [(metrics[class_names[0]][m]['ci_high'] - metrics[class_names[0]][m]['ci_low'])/2 for m in metric_names]
    neg_errs = [(metrics[class_names[1]][m]['ci_high'] - metrics[class_names[1]][m]['ci_low'])/2 for m in metric_names]
    
    y_pos = np.arange(len(metric_names))
    width = 0.35
    axes[1,0].barh(y_pos - width/2, pos_means, width, xerr=pos_errs, label=class_names[0], color='#E31A1C', alpha=0.8, capsize=3, hatch='///')
    axes[1,0].barh(y_pos + width/2, neg_means, width, xerr=neg_errs, label=class_names[1], color='#1F78B4', alpha=0.8, capsize=3, hatch='...')
    
    for i in range(len(metric_names)):
        axes[1,0].text(pos_means[i]+pos_errs[i]+0.02, y_pos[i]-width/2, f'{pos_means[i]:.3f}', va='center', fontsize=10)
        axes[1,0].text(neg_means[i]+neg_errs[i]+0.02, y_pos[i]+width/2, f'{neg_means[i]:.3f}', va='center', fontsize=10)
    
    axes[1,0].set_yticks(y_pos)
    axes[1,0].set_yticklabels(metric_names)
    axes[1,0].set_xlabel("Score", fontsize=fontsize_labels)
    axes[1,0].set_title("C. Classification Metrics", fontsize=fontsize_title, fontweight='bold')
    axes[1,0].tick_params(axis='both', which='major', labelsize=fontsize_ticks)
    axes[1,0].legend(fontsize=fontsize_legend)
    axes[1,0].grid(True, alpha=0.3, axis='x')
    axes[1,0].set_xlim(0, 1.15)
    

    labels = [neg_label, pos_label]
    names = [class_info['neg_name'], class_info['pos_name']]
    cm = confusion_matrix(y_test, y_pred, labels=labels)
    cm_pct = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis] * 100
    sns.heatmap(cm, annot=False, cmap='Blues', xticklabels=names, yticklabels=names, ax=axes[1,1])
    
    for i in range(2):
        for j in range(2):
            color = 'white' if cm[i,j] > cm.max()/2 else 'black'
            axes[1,1].text(j+0.5, i+0.5, f'{cm[i,j]:,}\n({cm_pct[i,j]:.1f}%)', ha='center', va='center', color=color, fontweight='bold')
    
    axes[1,1].set_title("D. Confusion Matrix", fontsize=fontsize_title, fontweight='bold')
    axes[1,1].set_xlabel("Predicted Labels", fontsize=fontsize_labels)
    axes[1,1].set_ylabel("True Labels", fontsize=fontsize_labels)
    axes[1,1].tick_params(axis='both', which='major', labelsize=fontsize_ticks)
    
    tn, fp, fn, tp = cm.ravel()
    acc = (tp + tn)/cm.sum()
    sens = tp/(tp+fn) if (tp+fn)>0 else 0
    spec = tn/(tn+fp) if (tn+fp)>0 else 0
    axes[1,1].text(0.5, -0.18, f'Accuracy: {acc:.3f} | Sensitivity: {sens:.3f} | Specificity: {spec:.3f}', 
                    transform=axes[1,1].transAxes, ha='center', style='italic', fontsize=12)
    
    fig.suptitle(f"{model_name} Evaluation", fontsize=20, fontweight='bold')
    plt.tight_layout(rect=[0,0,1,0.96])
    

    fig.savefig(f"{save_dir}/figures/{prefix}_2x2_grid.png", dpi=1200, bbox_inches='tight')
    fig.savefig(f"{save_dir}/figures/{prefix}_2x2_grid.pdf", bbox_inches='tight')
    plt.close(fig)
    

    fig_cal = plot_calibration(y_test, y_prob, y_prob_raw, class_info)
    fig_cal.savefig(f"{save_dir}/figures/{prefix}_calibration.png", dpi=1200, bbox_inches='tight')
    fig_cal.savefig(f"{save_dir}/figures/{prefix}_calibration.pdf", bbox_inches='tight')
    plt.close(fig_cal)
    

    return {
        'y_pred': y_pred,
        'y_prob': y_prob,
        'auc': auc_ci,
        'auprc_pos': ap_ci_pos,
        'auprc_neg': ap_ci_neg,
        'metrics_table': metrics_table,
        'fig_grid': fig,
        'fig_calibration': fig_cal
    }

evaluate_and_plot(
    model=xgb_calibrated_model,         
    X_test=X_test_reduced,             
    y_test=y_test_set_encoded,         
    le=le,                             
    save_dir="../results/XGB",  
    prefix="XGB",                        
    model_name="XGBoost",        
    raw_model=xgb_bayes_search          
)

In [ ]:
# XGBoost Classifier Explainability with SHAP

def explain_model_with_shap_plots(
    pipeline, X_train, X_test, save_dir, classifier_step_name="classifier", 
    sample_idx=0, only_explain_gc=True
):
    """
    Generates SHAP global summary plot and SHAP waterfall plot for a specific sample.
    """
    # Set larger font sizes for all plots
    plt.rcParams.update({
        'font.size': 14,
        'axes.titlesize': 16,
        'axes.labelsize': 14,
        'xtick.labelsize': 12,
        'ytick.labelsize': 12,
        'legend.fontsize': 12
    })

    os.makedirs(os.path.join(save_dir, "figures"), exist_ok=True)

    # Convert categorical variables to numeric
    X_train_num = X_train.copy()
    X_test_num = X_test.copy()
    
    for col in X_train_num.select_dtypes(include="category").columns:
        X_train_num[col] = X_train_num[col].cat.codes
        X_test_num[col] = X_test_num[col].cat.codes

    X_train_num = X_train_num.astype(float)
    X_test_num = X_test_num.astype(float)
    
    # Handle problematic values
    X_train_num = X_train_num.replace([np.inf, -np.inf], np.nan)
    X_test_num = X_test_num.replace([np.inf, -np.inf], np.nan)
    X_train_num = X_train_num.fillna(X_train_num.median())
    X_test_num = X_test_num.fillna(X_train_num.median())

    # Auto-detect classifier step
    if hasattr(pipeline, "named_steps"):  
        model_step_names = list(pipeline.named_steps.keys())
        classifier_step_name = model_step_names[-1]
        model = pipeline.named_steps[classifier_step_name]
    elif hasattr(pipeline, "estimator") and hasattr(pipeline.estimator, "named_steps"):  
        model_step_names = list(pipeline.estimator.named_steps.keys())
        classifier_step_name = model_step_names[-1]
        model = pipeline.estimator.named_steps[classifier_step_name]
    else:
        raise ValueError("Could not find classifier step in pipeline.")

    print(f"Using classifier: {classifier_step_name} ({type(model).__name__})")

    # Select appropriate SHAP explainer
    if isinstance(model, (RandomForestClassifier, xgb.XGBClassifier)):
        explainer = shap.TreeExplainer(model)
    elif isinstance(model, LogisticRegression):
        background_sample = shap.sample(X_train_num, 100)
        explainer = shap.LinearExplainer(model, background_sample)
    else:
        background_sample = shap.sample(X_train_num, 50)
        explainer = shap.PermutationExplainer(model.predict, background_sample)

    # Calculate SHAP values
    print("Computing SHAP values...")
    shap_values = explainer(X_test_num)

    if sample_idx >= len(X_test_num):
        sample_idx = 0
    
    sample_pred = pipeline.predict(X_test_num.iloc[[sample_idx]])[0]
    sample_probs = pipeline.predict_proba(X_test_num.iloc[[sample_idx]])[0]

    predicted_class = "Gastric cancer" if sample_pred == 0 else "Non-gastric cancer"
    print(f"Sample {sample_idx} predicted as: {predicted_class}")

    if only_explain_gc and sample_pred != 0:
        print(f"Skipping explanation: Sample not classified as gastric cancer.")
        return

    if hasattr(shap_values, 'values'):
        if shap_values.values.ndim == 3:
            values_for_bar = shap_values.values[:, :, 0]
            waterfall_values = shap_values.values[sample_idx, :, 0]
            expected_value = shap_values.base_values[sample_idx, 0]
        else:
            values_for_bar = shap_values.values
            waterfall_values = shap_values.values[sample_idx]
            expected_value = shap_values.base_values[sample_idx]
        feature_values = shap_values.data
    else:
        values_for_bar = shap_values
        waterfall_values = shap_values[sample_idx]
        expected_value = explainer.expected_value
        feature_values = X_test_num.values

    # Create SHAP explanation object for bar plot
    shap_explanation_for_bar = shap.Explanation(
        values=values_for_bar,
        base_values=expected_value,
        data=feature_values,
        feature_names=X_test_num.columns.tolist()
    )

    # Create explanation object for waterfall plot
    explanation_obj = shap.Explanation(
        values=waterfall_values,
        base_values=expected_value,
        data=X_test_num.iloc[sample_idx].values,
        feature_names=X_test_num.columns.tolist()
    )

    # Create combined plot with subplots
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(20, 40), gridspec_kw={'hspace': 0.8})
    
    # Global SHAP bar plot
    ax1.text(-0.02, 1.25, 'A', transform=ax1.transAxes, fontsize=20, fontweight='bold', va='top')
    shap.plots.bar(shap_explanation_for_bar, ax=ax1, show=False, max_display=10)
    ax1.set_title('Global Feature Importance', fontsize=16, fontweight='bold', pad=20)
    
    # Individual waterfall plot 
    ax2.text(-0.02, 1.4, 'B', transform=ax2.transAxes, fontsize=20, fontweight='bold', va='top')
    plt.sca(ax2)
    shap.waterfall_plot(explanation_obj, show=False)
    ax2.set_title(f'Individual Prediction Explanation - Sample {sample_idx}', fontsize=16, fontweight='bold', pad=20)
    
    plt.tight_layout(pad=2.0)
    combined_path = os.path.join(save_dir, "figures", f"shap_combined_plots_sample_{sample_idx}.png")
    plt.savefig(combined_path, dpi=1200, bbox_inches="tight", facecolor="white")
    plt.close()
    print(f"Combined SHAP plots saved to: {combined_path}")


explain_model_with_shap_plots(
        xgb_calibrated_model, 
        X_train_reduced, 
        X_test_reduced, 
        "../results/XGB", 
        sample_idx=2,
        only_explain_gc=True
    )

Using classifier: classifier (XGBClassifier)
Computing SHAP values...
Sample 2 predicted as: Gastric cancer
Combined SHAP plots saved to: ../results/XGB/figures/shap_combined_plots_sample_2.png


In [24]:
# Random Forest Classifier

rf_pipeline = Pipeline([('smotenc', SMOTENC(categorical_features=categorical_indices_subset, random_state=26)),
    ('classifier_rf', RandomForestClassifier(random_state=26))
])

params_rf_bayes = {
    'classifier_rf__n_estimators': Integer(50, 2000),
    'classifier_rf__criterion': Categorical(['gini', 'entropy', 'log_loss']),
    'classifier_rf__max_depth': Integer(3, 10),
    'classifier_rf__min_samples_split': Integer(2, 10),
    'classifier_rf__min_samples_leaf': Integer(1, 10),
    'classifier_rf__bootstrap': Categorical([True, False]),
    'classifier_rf__ccp_alpha': Real(1e-6, 0.01, prior='log-uniform'),
}

bayes_search_rf = BayesSearchCV(
    estimator=rf_pipeline,
    search_spaces=params_rf_bayes,
    scoring='recall',
    cv=ss,
    n_iter=60,
    n_points=2,
    n_jobs=2,
    verbose=2,
    random_state=26
)
# Fit the model
bayes_search_rf.fit(X_train_reduced, y_train_set_encoded)


rf_calibrated_model = CalibratedClassifierCV(bayes_search_rf.best_estimator_, method='sigmoid', cv=ss)
rf_calibrated_model.fit(X_train_reduced, y_train_set_encoded)

Fitting 10 folds for each of 2 candidates, totalling 20 fits
[CV] END classifier_rf__bootstrap=False, classifier_rf__ccp_alpha=0.008916481566215747, classifier_rf__criterion=gini, classifier_rf__max_depth=5, classifier_rf__min_samples_leaf=6, classifier_rf__min_samples_split=10, classifier_rf__n_estimators=1075; total time=   8.5s
[CV] END classifier_rf__bootstrap=False, classifier_rf__ccp_alpha=0.008916481566215747, classifier_rf__criterion=gini, classifier_rf__max_depth=5, classifier_rf__min_samples_leaf=6, classifier_rf__min_samples_split=10, classifier_rf__n_estimators=1075; total time=   8.6s
[CV] END classifier_rf__bootstrap=False, classifier_rf__ccp_alpha=0.008916481566215747, classifier_rf__criterion=gini, classifier_rf__max_depth=5, classifier_rf__min_samples_leaf=6, classifier_rf__min_samples_split=10, classifier_rf__n_estimators=1075; total time=   8.6s
[CV] END classifier_rf__bootstrap=False, classifier_rf__ccp_alpha=0.008916481566215747, classifier_rf__criterion=gini, clas

CalibratedClassifierCV(cv=StratifiedShuffleSplit(n_splits=10, random_state=26, test_size=None,
            train_size=None),
                       estimator=Pipeline(steps=[('smotenc',
                                                  SMOTENC(categorical_features=[92],
                                                          random_state=26)),
                                                 ('classifier_rf',
                                                  RandomForestClassifier(bootstrap=False,
                                                                         ccp_alpha=8.863227160496232e-06,
                                                                         max_depth=10,
                                                                         min_samples_split=3,
                                                                         n_estimators=955,
                                                                         random_state=26))]))

In [ ]:
# Random Forest Classifier Evaluation

plt.rcParams.update({           
    'font.size': 8, 'axes.titlesize': 9, 'axes.labelsize': 8,
    'xtick.labelsize': 7, 'ytick.labelsize': 7, 'legend.fontsize': 7,
    'figure.dpi': 1200, 'savefig.dpi': 1200
})

def get_class_info(le, pos_class="Gastric cancer"):
    """Figure out which label is which class"""
    mapping = {name: i for i, name in enumerate(le.classes_)}
    neg_class = [k for k in mapping.keys() if k != pos_class][0]
    
    return {
        'pos_name': pos_class, 'pos_label': mapping[pos_class],
        'neg_name': neg_class, 'neg_label': mapping[neg_class],
        'mapping': mapping
    }

def bootstrap_ci(y_true, y_pred, y_prob, metric_func, n=1000):
    """Get confidence intervals via bootstrap"""
    np.random.seed(26)  # reproducibility
    scores = []
    
    for _ in range(n):
        idx = np.random.choice(len(y_true), len(y_true), replace=True)
        
        # Skip if only one class in bootstrap sample
        if len(np.unique(y_true[idx])) < 2:
            continue
            
        try:
            if y_prob is not None:
                score = metric_func(y_true[idx], y_prob[idx])
            else:
                score = metric_func(y_true[idx], y_pred[idx])
            scores.append(score)
        except:
            continue  # skip problematic samples
    
    if not scores:
        return {'mean': np.nan, 'ci_low': np.nan, 'ci_high': np.nan}
    
    scores = np.array(scores)
    return {
        'mean': np.mean(scores),
        'ci_low': np.percentile(scores, 2.5),
        'ci_high': np.percentile(scores, 97.5)
    }

def plot_calibration(y_test, y_prob_cal, y_prob_raw, class_info, title="Calibration"):
    """Before/after calibration plots"""
    pos_label = class_info['pos_label']
    y_true_binary = (y_test == pos_label).astype(int)
    
    # FIXED: Handle axes creation properly
    if y_prob_raw is not None:
        fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    else:
        fig, ax = plt.subplots(1, 1, figsize=(5, 4))
        axes = [ax]
    
    plot_idx = 0
    
    # Before calibration
    if y_prob_raw is not None:
        frac, pred = calibration_curve(y_true_binary, y_prob_raw[:, pos_label], n_bins=10)
        brier = brier_score_loss(y_true_binary, y_prob_raw[:, pos_label])
        
        axes[plot_idx].plot(pred, frac, 's-', label=class_info['pos_name'])
        axes[plot_idx].plot([0, 1], [0, 1], 'k:', label="Perfect")
        axes[plot_idx].set_title("Before Calibration")
        axes[plot_idx].text(0.02, 0.98, f'Brier Score: {brier:.3f}', 
                           transform=axes[plot_idx].transAxes, va='top',
                           bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
        axes[plot_idx].legend()
        plot_idx += 1
    
    # After calibration
    frac, pred = calibration_curve(y_true_binary, y_prob_cal[:, pos_label], n_bins=10)
    brier = brier_score_loss(y_true_binary, y_prob_cal[:, pos_label])
    
    axes[plot_idx].plot(pred, frac, 's-', label=class_info['pos_name'])
    axes[plot_idx].plot([0, 1], [0, 1], 'k:', label="Perfect")
    axes[plot_idx].set_title("After Calibration" if y_prob_raw is not None else "Calibration")
    axes[plot_idx].text(0.02, 0.98, f'Brier Score: {brier:.3f}', 
                       transform=axes[plot_idx].transAxes, va='top',
                       bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
    axes[plot_idx].legend()
    
    # FIXED: Now axes is always a list, so this works correctly
    for ax in axes:
        ax.set_xlabel("Mean Predicted Probability")
        ax.set_ylabel("Fraction of Positives")
        ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    return fig

def create_metrics_table(y_test, y_pred, y_prob, class_info):
    """Create comprehensive metrics table with confidence intervals"""
    pos_label = class_info['pos_label']
    neg_label = class_info['neg_label']
    pos_name = class_info['pos_name']
    neg_name = class_info['neg_name']
    
    # Calculate metrics with CIs for both classes
    metrics_data = []
    
    # Per-class metrics
    for label, class_name in [(pos_label, pos_name), (neg_label, neg_name)]:
        # Precision
        prec_ci = bootstrap_ci(y_test, y_pred, None, 
                              lambda yt, yp: precision_score(yt, yp, pos_label=label, zero_division=0))
        # Recall (Sensitivity for pos class, Specificity calculation for neg class)
        rec_ci = bootstrap_ci(y_test, y_pred, None, 
                             lambda yt, yp: recall_score(yt, yp, pos_label=label, zero_division=0))
        # F1-Score
        f1_ci = bootstrap_ci(y_test, y_pred, None, 
                            lambda yt, yp: f1_score(yt, yp, pos_label=label, zero_division=0))
        
        # AUPRC for this class
        def ap_func_class(y_true, y_prob):
            return average_precision_score(y_true, y_prob, pos_label=label)
        
        ap_ci = bootstrap_ci(y_test, None, y_prob[:, label], ap_func_class)
        
        metrics_data.extend([
            {
                'Class': class_name,
                'Metric': 'Precision',
                'Value': prec_ci['mean'],
                'CI_Lower': prec_ci['ci_low'],
                'CI_Upper': prec_ci['ci_high'],
                'Final': f"{prec_ci['mean']:.3f} ({prec_ci['ci_low']:.3f}–{prec_ci['ci_high']:.3f})"
            },
            {
                'Class': class_name,
                'Metric': 'Recall',
                'Value': rec_ci['mean'],
                'CI_Lower': rec_ci['ci_low'],
                'CI_Upper': rec_ci['ci_high'],
                'Final': f"{rec_ci['mean']:.3f} ({rec_ci['ci_low']:.3f}–{rec_ci['ci_high']:.3f})"
            },
            {
                'Class': class_name,
                'Metric': 'F1-Score',
                'Value': f1_ci['mean'],
                'CI_Lower': f1_ci['ci_low'],
                'CI_Upper': f1_ci['ci_high'],
                'Final': f"{f1_ci['mean']:.3f} ({f1_ci['ci_low']:.3f}–{f1_ci['ci_high']:.3f})"
            },
            {
                'Class': class_name,
                'Metric': 'AUPRC',
                'Value': ap_ci['mean'],
                'CI_Lower': ap_ci['ci_low'],
                'CI_Upper': ap_ci['ci_high'],
                'Final': f"{ap_ci['mean']:.3f} ({ap_ci['ci_low']:.3f}–{ap_ci['ci_high']:.3f})"
            }
        ])
    
    # Overall metrics
    # Accuracy
    acc_ci = bootstrap_ci(y_test, y_pred, None, lambda yt, yp: accuracy_score(yt, yp))
    
    # AUC-ROC (for positive class)
    def auc_func(y_true, y_prob):
        fpr, tpr, _ = roc_curve(y_true, y_prob, pos_label=pos_label)
        return auc(fpr, tpr)
    auc_ci = bootstrap_ci(y_test, None, y_prob[:, pos_label], auc_func)
    
    # Add overall metrics
    overall_metrics = [
        {
            'Class': 'Overall',
            'Metric': 'Accuracy',
            'Value': acc_ci['mean'],
            'CI_Lower': acc_ci['ci_low'],
            'CI_Upper': acc_ci['ci_high'],
            'Final': f"{acc_ci['mean']:.3f} ({acc_ci['ci_low']:.3f}–{acc_ci['ci_high']:.3f})"
        },
        {
            'Class': 'Overall',
            'Metric': 'AUC-ROC',
            'Value': auc_ci['mean'],
            'CI_Lower': auc_ci['ci_low'],
            'CI_Upper': auc_ci['ci_high'],
            'Final': f"{auc_ci['mean']:.3f} ({auc_ci['ci_low']:.3f}–{auc_ci['ci_high']:.3f})"
        }
    ]
    
    metrics_data.extend(overall_metrics)
    
    # Create DataFrame
    df = pd.DataFrame(metrics_data)
    
    return df

def evaluate_and_plot(model, X_test, y_test, le, save_dir, prefix, 
                      model_name="Model", raw_model=None):
    """
    Full evaluation pipeline:
    - Predictions
    - Bootstrap CIs
    - Metrics table
    - CSV outputs
    - 2x2 grid figure (ROC, PR, metrics, confusion matrix)
    - Calibration plot (separate)
    """
    os.makedirs(f"{save_dir}/figures", exist_ok=True)
    
    class_info = get_class_info(le)
    pos_label = class_info['pos_label']
    neg_label = class_info['neg_label']
    
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)
    y_prob_raw = raw_model.predict_proba(X_test) if raw_model else None
    
    y_test_orig = le.inverse_transform(y_test)
    y_pred_orig = le.inverse_transform(y_pred)
    report = classification_report(y_test_orig, y_pred_orig, output_dict=True)
    pd.DataFrame(report).T.to_csv(f"{save_dir}/{prefix}_report.csv")
    
    def auc_func(y_true, y_prob): return auc(*roc_curve(y_true, y_prob, pos_label=pos_label)[:2])
    def ap_func_pos(y_true, y_prob): return average_precision_score(y_true, y_prob, pos_label=pos_label)
    def ap_func_neg(y_true, y_prob): return average_precision_score(y_true, y_prob, pos_label=neg_label)
    
    auc_ci = bootstrap_ci(y_test, None, y_prob[:, pos_label], auc_func)
    ap_ci_pos = bootstrap_ci(y_test, None, y_prob[:, pos_label], ap_func_pos)
    ap_ci_neg = bootstrap_ci(y_test, None, y_prob[:, neg_label], ap_func_neg)
    
    metrics_table = create_metrics_table(y_test, y_pred, y_prob, class_info)
    metrics_table.to_csv(f"{save_dir}/{prefix}_detailed_metrics.csv", index=False)
    
    fig, axes = plt.subplots(2, 2, figsize=(18, 14))
    plt.subplots_adjust(hspace=0.7, wspace=0.7)

    fontsize_labels = 16   
    fontsize_ticks = 14    
    fontsize_legend = 14   
    fontsize_title = 18    
    
    fpr, tpr, _ = roc_curve(y_test, y_prob[:, pos_label], pos_label=pos_label)
    axes[0,0].plot(fpr, tpr, 'r-', linewidth=2,
                   label=f"{class_info['pos_name']} - AUC = {auc_ci['mean']:.3f} "
                         f"(95% CI: {auc_ci['ci_low']:.3f}–{auc_ci['ci_high']:.3f})")
    axes[0,0].plot([0,1],[0,1],'k--', alpha=0.5)
    axes[0,0].set_xlabel("False Positive Rate", fontsize=fontsize_labels)
    axes[0,0].set_ylabel("True Positive Rate", fontsize=fontsize_labels)
    axes[0,0].tick_params(axis='both', which='major', labelsize=fontsize_ticks)
    axes[0,0].set_title("A. ROC Curve", fontsize=fontsize_title, fontweight='bold')
    axes[0,0].legend(fontsize=fontsize_legend)
    axes[0,0].grid(True, alpha=0.3)
    

    prec, rec, _ = precision_recall_curve(y_test, y_prob[:, pos_label], pos_label=pos_label)
    prec_neg, rec_neg, _ = precision_recall_curve(y_test, y_prob[:, neg_label], pos_label=neg_label)
    axes[0,1].plot(rec, prec, 'b-', linewidth=2,
                   label=f"{class_info['pos_name']} - AUPRC = {ap_ci_pos['mean']:.3f} "
                         f"(95% CI: {ap_ci_pos['ci_low']:.3f}–{ap_ci_pos['ci_high']:.3f})")
    axes[0,1].plot(rec_neg, prec_neg, 'g--', linewidth=2,
                   label=f"{class_info['neg_name']} - AUPRC = {ap_ci_neg['mean']:.3f} "
                         f"(95% CI: {ap_ci_neg['ci_low']:.3f}–{ap_ci_neg['ci_high']:.3f})")
    axes[0,1].set_xlabel("Recall", fontsize=fontsize_labels)
    axes[0,1].set_ylabel("Precision", fontsize=fontsize_labels)
    axes[0,1].set_title("B. Precision-Recall Curve", fontsize=fontsize_title, fontweight='bold')
    axes[0,1].tick_params(axis='both', which='major', labelsize=fontsize_ticks)
    axes[0,1].legend(fontsize=fontsize_legend)
    axes[0,1].grid(True, alpha=0.3)
    

    metrics = {}
    for label, name in [(pos_label, class_info['pos_name']), (neg_label, class_info['neg_name'])]:
        metrics[name] = {
            'Precision': bootstrap_ci(y_test, y_pred, None, lambda yt, yp: precision_score(yt, yp, pos_label=label, zero_division=0)),
            'Recall': bootstrap_ci(y_test, y_pred, None, lambda yt, yp: recall_score(yt, yp, pos_label=label, zero_division=0)),
            'F1-Score': bootstrap_ci(y_test, y_pred, None, lambda yt, yp: f1_score(yt, yp, pos_label=label, zero_division=0))
        }
    metric_names = ['Precision', 'Recall', 'F1-Score']
    class_names = list(metrics.keys())
    pos_means = [metrics[class_names[0]][m]['mean'] for m in metric_names]
    neg_means = [metrics[class_names[1]][m]['mean'] for m in metric_names]
    pos_errs = [(metrics[class_names[0]][m]['ci_high'] - metrics[class_names[0]][m]['ci_low'])/2 for m in metric_names]
    neg_errs = [(metrics[class_names[1]][m]['ci_high'] - metrics[class_names[1]][m]['ci_low'])/2 for m in metric_names]
    
    y_pos = np.arange(len(metric_names))
    width = 0.35
    axes[1,0].barh(y_pos - width/2, pos_means, width, xerr=pos_errs, label=class_names[0], color='#E31A1C', alpha=0.8, capsize=3, hatch='///')
    axes[1,0].barh(y_pos + width/2, neg_means, width, xerr=neg_errs, label=class_names[1], color='#1F78B4', alpha=0.8, capsize=3, hatch='...')
    
    for i in range(len(metric_names)):
        axes[1,0].text(pos_means[i]+pos_errs[i]+0.02, y_pos[i]-width/2, f'{pos_means[i]:.3f}', va='center', fontsize=10)
        axes[1,0].text(neg_means[i]+neg_errs[i]+0.02, y_pos[i]+width/2, f'{neg_means[i]:.3f}', va='center', fontsize=10)
    
    axes[1,0].set_yticks(y_pos)
    axes[1,0].set_yticklabels(metric_names)
    axes[1,0].set_xlabel("Score", fontsize=fontsize_labels)
    axes[1,0].set_title("C. Classification Metrics", fontsize=fontsize_title, fontweight='bold')
    axes[1,0].tick_params(axis='both', which='major', labelsize=fontsize_ticks)
    axes[1,0].legend(fontsize=fontsize_legend)
    axes[1,0].grid(True, alpha=0.3, axis='x')
    axes[1,0].set_xlim(0, 1.15)
    

    labels = [neg_label, pos_label]
    names = [class_info['neg_name'], class_info['pos_name']]
    cm = confusion_matrix(y_test, y_pred, labels=labels)
    cm_pct = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis] * 100
    sns.heatmap(cm, annot=False, cmap='Blues', xticklabels=names, yticklabels=names, ax=axes[1,1])
    
    for i in range(2):
        for j in range(2):
            color = 'white' if cm[i,j] > cm.max()/2 else 'black'
            axes[1,1].text(j+0.5, i+0.5, f'{cm[i,j]:,}\n({cm_pct[i,j]:.1f}%)', ha='center', va='center', color=color, fontweight='bold')
    
    axes[1,1].set_title("D. Confusion Matrix", fontsize=fontsize_title, fontweight='bold')
    axes[1,1].set_xlabel("Predicted Labels", fontsize=fontsize_labels)
    axes[1,1].set_ylabel("True Labels", fontsize=fontsize_labels)
    axes[1,1].tick_params(axis='both', which='major', labelsize=fontsize_ticks)
    
    tn, fp, fn, tp = cm.ravel()
    acc = (tp + tn)/cm.sum()
    sens = tp/(tp+fn) if (tp+fn)>0 else 0
    spec = tn/(tn+fp) if (tn+fp)>0 else 0
    axes[1,1].text(0.5, -0.18, f'Accuracy: {acc:.3f} | Sensitivity: {sens:.3f} | Specificity: {spec:.3f}', 
                    transform=axes[1,1].transAxes, ha='center', style='italic', fontsize=12)
    
    fig.suptitle(f"{model_name} Evaluation", fontsize=20, fontweight='bold')
    plt.tight_layout(rect=[0,0,1,0.96])
    

    fig.savefig(f"{save_dir}/figures/{prefix}_2x2_grid.png", dpi=1200, bbox_inches='tight')
    fig.savefig(f"{save_dir}/figures/{prefix}_2x2_grid.pdf", bbox_inches='tight')
    plt.close(fig)
    

    fig_cal = plot_calibration(y_test, y_prob, y_prob_raw, class_info)
    fig_cal.savefig(f"{save_dir}/figures/{prefix}_calibration.png", dpi=1200, bbox_inches='tight')
    fig_cal.savefig(f"{save_dir}/figures/{prefix}_calibration.pdf", bbox_inches='tight')
    plt.close(fig_cal)
    

    return {
        'y_pred': y_pred,
        'y_prob': y_prob,
        'auc': auc_ci,
        'auprc_pos': ap_ci_pos,
        'auprc_neg': ap_ci_neg,
        'metrics_table': metrics_table,
        'fig_grid': fig,
        'fig_calibration': fig_cal
    }

evaluate_and_plot(
    model=rf_calibrated_model,         
    X_test=X_test_reduced,             
    y_test=y_test_set_encoded,         
    le=le,                             
    save_dir="../results/RF",  
    prefix="RF",                        
    model_name="Random Forest",        
    raw_model=bayes_search_rf          
)

In [ ]:
# Random Forest Classifier Explanation with SHAP

def explain_model_with_shap_plots(
    pipeline, X_train, X_test, save_dir, classifier_step_name="classifier", 
    sample_idx=0, only_explain_gc=True
):
    """
    Generates SHAP global summary plot and SHAP waterfall plot for a specific sample.
    """
    # Set larger font sizes for all plots
    plt.rcParams.update({
        'font.size': 14,
        'axes.titlesize': 16,
        'axes.labelsize': 14,
        'xtick.labelsize': 12,
        'ytick.labelsize': 12,
        'legend.fontsize': 12
    })

    os.makedirs(os.path.join(save_dir, "figures"), exist_ok=True)

    # Convert categorical variables to numeric
    X_train_num = X_train.copy()
    X_test_num = X_test.copy()
    
    for col in X_train_num.select_dtypes(include="category").columns:
        X_train_num[col] = X_train_num[col].cat.codes
        X_test_num[col] = X_test_num[col].cat.codes

    X_train_num = X_train_num.astype(float)
    X_test_num = X_test_num.astype(float)
    
    # Handle problematic values
    X_train_num = X_train_num.replace([np.inf, -np.inf], np.nan)
    X_test_num = X_test_num.replace([np.inf, -np.inf], np.nan)
    X_train_num = X_train_num.fillna(X_train_num.median())
    X_test_num = X_test_num.fillna(X_train_num.median())

    # Auto-detect classifier step
    if hasattr(pipeline, "named_steps"):  
        model_step_names = list(pipeline.named_steps.keys())
        classifier_step_name = model_step_names[-1]
        model = pipeline.named_steps[classifier_step_name]
    elif hasattr(pipeline, "estimator") and hasattr(pipeline.estimator, "named_steps"):  
        model_step_names = list(pipeline.estimator.named_steps.keys())
        classifier_step_name = model_step_names[-1]
        model = pipeline.estimator.named_steps[classifier_step_name]
    else:
        raise ValueError("Could not find classifier step in pipeline.")

    print(f"Using classifier: {classifier_step_name} ({type(model).__name__})")

    # Select appropriate SHAP explainer
    if isinstance(model, (RandomForestClassifier, xgb.XGBClassifier)):
        explainer = shap.TreeExplainer(model)
    elif isinstance(model, LogisticRegression):
        background_sample = shap.sample(X_train_num, 100)
        explainer = shap.LinearExplainer(model, background_sample)
    else:
        background_sample = shap.sample(X_train_num, 50)
        explainer = shap.PermutationExplainer(model.predict, background_sample)

    # Calculate SHAP values
    print("Computing SHAP values...")
    shap_values = explainer(X_test_num)

    if sample_idx >= len(X_test_num):
        sample_idx = 0
    
    sample_pred = pipeline.predict(X_test_num.iloc[[sample_idx]])[0]
    sample_probs = pipeline.predict_proba(X_test_num.iloc[[sample_idx]])[0]

    predicted_class = "Gastric cancer" if sample_pred == 0 else "Non-gastric cancer"
    print(f"Sample {sample_idx} predicted as: {predicted_class}")

    if only_explain_gc and sample_pred != 0:
        print(f"Skipping explanation: Sample not classified as gastric cancer.")
        return

    if hasattr(shap_values, 'values'):
        if shap_values.values.ndim == 3:
            values_for_bar = shap_values.values[:, :, 0]
            waterfall_values = shap_values.values[sample_idx, :, 0]
            expected_value = shap_values.base_values[sample_idx, 0]
        else:
            values_for_bar = shap_values.values
            waterfall_values = shap_values.values[sample_idx]
            expected_value = shap_values.base_values[sample_idx]
        feature_values = shap_values.data
    else:
        values_for_bar = shap_values
        waterfall_values = shap_values[sample_idx]
        expected_value = explainer.expected_value
        feature_values = X_test_num.values

    # Create SHAP explanation object for bar plot
    shap_explanation_for_bar = shap.Explanation(
        values=values_for_bar,
        base_values=expected_value,
        data=feature_values,
        feature_names=X_test_num.columns.tolist()
    )

    # Create explanation object for waterfall plot
    explanation_obj = shap.Explanation(
        values=waterfall_values,
        base_values=expected_value,
        data=X_test_num.iloc[sample_idx].values,
        feature_names=X_test_num.columns.tolist()
    )

    # Create combined plot with subplots
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(20, 40), gridspec_kw={'hspace': 0.8})
    
    # Global SHAP bar plot
    ax1.text(-0.02, 1.25, 'A', transform=ax1.transAxes, fontsize=20, fontweight='bold', va='top')
    shap.plots.bar(shap_explanation_for_bar, ax=ax1, show=False, max_display=10)
    ax1.set_title('Global Feature Importance', fontsize=16, fontweight='bold', pad=20)
    
    # Individual waterfall plot 
    ax2.text(-0.02, 1.4, 'B', transform=ax2.transAxes, fontsize=20, fontweight='bold', va='top')
    plt.sca(ax2)  
    shap.waterfall_plot(explanation_obj, show=False)
    ax2.set_title(f'Individual Prediction Explanation - Sample {sample_idx}', fontsize=16, fontweight='bold', pad=20)
    
    plt.tight_layout(pad=2.0)
    combined_path = os.path.join(save_dir, "figures", f"shap_combined_plots_sample_{sample_idx}.png")
    plt.savefig(combined_path, dpi=1200, bbox_inches="tight", facecolor="white")
    plt.close()
    print(f"Combined SHAP plots saved to: {combined_path}")



explain_model_with_shap_plots(
    rf_calibrated_model, 
    X_train_reduced, 
    X_test_reduced, 
    "../results/RF", 
    sample_idx=2,
    only_explain_gc=True
)

Using classifier: classifier_rf (RandomForestClassifier)
Computing SHAP values...
Sample 2 predicted as: Gastric cancer
Combined SHAP plots saved to: ../results/RF/figures/shap_combined_plots_sample_2.png


In [70]:
# DeLong's test for comparing AUCs
from MLstatkit.stats import Delong_test

true = y_test_set_encoded
prob_A = xgb_calibrated_model.predict_proba(X_test_reduced)[:, 0] # Gastric Cancer
prob_B = lr_pipeline.predict_proba(X_test_set_encoded)[:, 0] # Gastric Cancer

# DeLong's test
z_score, p_value = Delong_test(true, prob_A, prob_B)
print(f"Z-score: {z_score}, P-value: {p_value}")

Z-score: -2.9633452739711297, P-value: 0.003043150098698074


In [71]:
# DeLong's test for comparing AUCs
from MLstatkit.stats import Delong_test

true = y_test_set_encoded
prob_A = rf_calibrated_model.predict_proba(X_test_reduced)[:, 0] # Gastric Cancer
prob_B = lr_pipeline.predict_proba(X_test_set_encoded)[:, 0] # Gastric Cancer

# DeLong's test
z_score, p_value = Delong_test(true, prob_A, prob_B)
print(f"Z-score: {z_score}, P-value: {p_value}")

Z-score: -3.2085019288115304, P-value: 0.001334284142251791
